#SECTION A – THEORETICAL QUESTIONS

Q1. What are the most common reasons for missing data in ETL pipelines?

Here are some most common reason for missing data in ETL Pipelines:

● User Error: Form not filled properly.

● System/Software Error: Failed API calls, faulty
sensors.

● Optional Fields: Users allowed to skip questions.

● Data Integration Issues: Format mismatches,
encoding issues.

● Human Errors: Typos, mis-entries, wrong file
handling.

Q2. Why is blindly deleting rows with missing values considered a bad practice in ETL?

Blindly deleting rows with missing values (Listwise Deletion) is a "silent killer" of data integrity for several critical reasons:

1. Introduction of Selection Bias
Missingness is rarely random. If data is missing because of a specific condition—for example, lower-income users being less likely to report salary—deleting those rows removes an entire demographic. Your resulting analysis will be skewed toward the "visible" population, leading to incorrect business conclusions.

2. Loss of Statistical Power
In large ETL pipelines, "missingness" can be spread across many columns. If 5% of data is missing in Column A and a different 5% in Column B, deleting rows with any nulls can quickly result in losing 30–50% of your total dataset. This reduces the sample size and makes your statistical results less reliable.

3. Destruction of Context
A NULL value is often informative data in itself.

Example: In a "Date of Cancellation" column, a missing value indicates an active customer. Deleting that row removes your most successful users from the pipeline.

4. Violation of Referential Integrity
If you delete a row in a "Sales" table because a non-critical "Discount Code" is missing, you may break the link to the "Customer" or "Product" tables. This creates "orphan" records elsewhere in your data warehouse that can no longer be joined or aggregated correctly.

Q3. Explain the difference between:

Listwise deletion

Column deletion

Also mention one scenario where each is appropriate.

In the world of data cleaning, these represent two different "surgical" approaches to handling missing data. One removes horizontal slices (rows), while the other removes vertical slices (columns).

1. Listwise Deletion (Row Removal)
Listwise deletion is the process of removing an entire row (record) if any single value within that row is missing.

How it works: If you have 50 columns of data and only 1 column is null for a specific user, the entire user record is purged from the dataset.

The Risk: It is the most aggressive form of data removal. It can significantly shrink your sample size and introduce "selection bias" if the data isn't missing completely at random.

When is it appropriate?
Listwise deletion is best when the missing value is in a Target Variable or a Primary Key.

Scenario: You are building a model to predict "Loan Default." If the "Default Status" (the thing you are trying to predict) is missing for a group of customers, those rows are useless for training your model and should be deleted.

2. Column Deletion (Feature Removal)
Column deletion involves removing an entire variable from the dataset across all records because it contains too many missing values to be reliable.

How it works: Instead of dropping users (rows), you drop the attribute (column). If "Middle Name" is missing for 90% of your customers, you simply remove the "Middle Name" column from your pipeline.

The Risk: You might throw away a variable that has high predictive power just because it’s hard to collect.

When is it appropriate?
Column deletion is best when a variable has a high "Missingness Ratio" (typically >60–70%) and is not critical to the analysis.

Scenario: You are analyzing website traffic. You have a column for "Referrer Social Media Handle," but 95% of your traffic comes from direct search or email, leaving that column almost entirely empty. Rather than trying to guess (impute) those handles, you drop the column to simplify the model.

Q4. Why is median imputation preferred over mean imputation for skewed data such as income?


Median imputation is preferred because it is resistant to outliers, whereas the mean is easily "pulled" by extreme values.

The Problem with Mean in Skewed Data
In a dataset like income, a few billionaires (outliers) will drastically inflate the mean. If you use that inflated mean to fill in missing values for average citizens, you create a "hallucination" of wealth that doesn't exist, leading to biased analysis.

The Advantage of Median
The median represents the true "middle" of the data distribution. It ignores how high the highest values are and how low the lowest values are, making it a much more accurate "typical" value for skewed distributions.

Q5. What is forward fill and in what type of dataset is it most useful?

Forward fill is a technique where you replace a missing value with the most recent non-null value from the same column.

If a value is missing at "Time 2," the pipeline simply "carries over" the value from "Time 1" until a new, valid data point is recorded.

It is most effective in Time-Series Datasets where observations are expected to remain constant or change gradually between recordings.

Q6. Why should flagging missing values be done before imputation in an ETL workflow?

Flagging missing values before imputation is essential because imputation erases the "signal" of missingness. Once you fill a NULL with a value like the median, you lose the ability to tell which data was real and which was estimated.

Q7. Consider a scenario where income is missing for many customers.

  How can this missingness itself provide business insights?

Scenario: High-income earners are hiding their data.(Not at Random)

Scenario: A technical bug in the "Income" field of your mobile app (AT Random).

When income is missing for many customers, the pattern of missingness itself can be highly informative. In business analytics, missing data is often a signal—not just a data problem.

Here’s how it can provide insights:

If many customers leave income blank:

They may be uncomfortable sharing financial information.

It may indicate low trust in the brand.

Your data collection process might feel intrusive.

insight: You may need better messaging on why income is requested or stronger trust-building strategies.

Missing income may not be random. It could correlate with:

Age group

Geography

Purchase behavior

Acquisition channel

For example:

High-value customers may avoid sharing income.

Lower-income customers may skip it due to sensitivity.

insight: Missingness can act as a behavioral segment indicator.

High missingness might indicate:

Poor form placement

Optional vs. required confusion

Friction in onboarding

Mobile UX problems

insight: Improving form design could increase data completeness and conversions.



Q8. Listwise Deletion

Remove all rows where Region is missing.

Tasks:

Identify affected rows

Show the dataset after deletion

Mention how many records were lost

Identify affected rows


In [ ]:
SELECT * FROM Customers
WHERE Region IS NULL;

Show the dataset after deletion


In [ ]:
DELETE FROM Customers WHERE Region IS NULL;

SELECT * FROM Customers;

Mention how many records were lost


In [ ]:
SELECT COUNT(*) AS Records_Lost FROM Customers WHERE Region IS NULL;

Q9. Imputation

Handle missing values in Monthly_Sales using:

Forward Fill

Tasks:

Apply forward fill

Show before vs after values

Explain why forward fill is suitable here

Apply forward fill

In [ ]:
SELECT
    Customer_ID,
    Name,
    Monthly_Sales AS Original_Sales,
    LAST_VALUE(Monthly_Sales IGNORE NULLS) OVER (ORDER BY Customer_ID) AS Forward_Filled_Sales
FROM Customers;

Show before vs after values

In [ ]:
SELECT
    Customer_ID,
    Name,
    Monthly_Sales AS Before_Value,
    LAST_VALUE(Monthly_Sales IGNORE NULLS) OVER (
        ORDER BY Customer_ID
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS After_Value
FROM Customers;

Explain why forward fill is suitable here

I applied the Forward Fill method to the Monthly_Sales column. This resulted in filling three missing values using the data from the preceding records (101, 103, and 105). This method is suitable here because it maintains sequential logic within the dataset, ensuring that missing values are estimated based on the most recent known observation rather than an overall average."

Q10. Flagging Missing Data

Create a flag column for missing Income.

Tasks:

Create Income_Missing_Flag (0 = present, 1 = missing)

Show updated dataset

Count how many customers have missing income

Create a flag column for missing Income.

In [ ]:
ALTER TABLE Customers ADD COLUMN Income_Missing_Flag INT;

Create Income_Missing_Flag (0 = present, 1 = missing)

In [ ]:
UPDATE Customers
SET Income_Missing_Flag = CASE
    WHEN Income IS NULL THEN 1
    ELSE 0
END;

Show updated dataset

In [ ]:
SELECT
    Customer_ID,
    Name,
    Income,
    Income_Missing_Flag
FROM Customers
ORDER BY Customer_ID;

Count how many customers have missing income


In [ ]:
SELECT COUNT(*) AS Total_Missing_Income
FROM Customers
WHERE Income_Missing_Flag = 1;